In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!

from pyspark.sql import functions as F

df_silver = spark.table("silver_ed_quarterly")

print("Silver records:", df_silver.count())
print("Silver columns:", len(df_silver.columns))


StatementMeta(, fe278902-f976-4def-994a-ba66bed75f22, 3, Finished, Available, Finished, False)

Silver records: 6930
Silver columns: 18


In [2]:

df_hospital = (
    df_silver
    .filter(F.col("triage_category") == "ALL")
    .select(
        "facility_code",
        "facility_name",
        "reporting_quarter",
        "quarter_end_month",
        F.col("number_of_attendances").alias("total_attendances"),
        "seen_within_target_pct",
        "median_wait_minutes",
        "did_not_wait_pct",
        "admitted_from_ed_pct",
        "admissions_within_4h_pct",
        "ed_stay_within_4h_pct",
        "source_file"
    )
)

print("Hospital-quarter records:", df_hospital.count())

display(
    df_hospital
    .filter(F.col("facility_name") == "Toowoomba Hospital")
    .orderBy("quarter_end_month")
)


StatementMeta(, fe278902-f976-4def-994a-ba66bed75f22, 4, Finished, Available, Finished, False)

Hospital-quarter records: 1155


SynapseWidget(Synapse.DataFrame, 8635f6f8-31eb-44d2-a784-481eff131a94)

In [3]:

# Ensure one record per facility and quarter
duplicates = (
    df_hospital
    .groupBy("facility_code", "quarter_end_month")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate hospital-quarter groups:", duplicates)

if duplicates != 0:
    raise ValueError("Duplicate hospital-quarter records detected")

if df_hospital.filter(
    F.col("quarter_end_month").isNull()
).count() > 0:
    raise ValueError("Invalid reporting dates detected")

# Save the Gold table
df_hospital.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_ed_hospital_quarterly")

print("Gold hospital table saved successfully!")


StatementMeta(, fe278902-f976-4def-994a-ba66bed75f22, 5, Finished, Available, Finished, False)

Duplicate hospital-quarter groups: 0
Gold hospital table saved successfully!


In [4]:

from pyspark.sql import functions as F

# Read cleaned Silver data
df_silver = spark.table("silver_ed_quarterly")

# Select only individual triage categories 1–5
df_triage = (
    df_silver
    .filter(
        F.col("triage_category").isin("1", "2", "3", "4", "5")
    )
    .select(
        "facility_code",
        "facility_name",
        "reporting_quarter",
        "quarter_end_month",
        "triage_category",
        F.col("number_of_attendances").alias("attendances"),
        "seen_within_target_pct",
        "median_wait_minutes",
        "did_not_wait_pct",
        "admitted_from_ed_pct",
        "admissions_within_4h_pct",
        "ed_stay_within_4h_pct",
        "source_file"
    )
)

# Add Australasian Triage Scale recommended times
df_triage = df_triage.withColumn(
    "recommended_time_minutes",
    F.when(F.col("triage_category") == "1", 0)
     .when(F.col("triage_category") == "2", 10)
     .when(F.col("triage_category") == "3", 30)
     .when(F.col("triage_category") == "4", 60)
     .when(F.col("triage_category") == "5", 120)
)

print("Triage records:", df_triage.count())
print("Total columns:", len(df_triage.columns))

display(
    df_triage
    .filter(F.col("facility_name") == "Toowoomba Hospital")
    .orderBy("quarter_end_month", "triage_category")
    .limit(15)
)


StatementMeta(, fe278902-f976-4def-994a-ba66bed75f22, 6, Finished, Available, Finished, False)

Triage records: 5775
Total columns: 14


SynapseWidget(Synapse.DataFrame, 12a4d028-be9b-4132-8d5b-7b3d96ea2104)

In [5]:

# Check duplicate facility-quarter-triage records
duplicate_groups = (
    df_triage
    .groupBy(
        "facility_code",
        "quarter_end_month",
        "triage_category"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Duplicate triage groups:", duplicate_groups)

# Check missing recommended treatment times
missing_targets = df_triage.filter(
    F.col("recommended_time_minutes").isNull()
).count()

print("Missing treatment targets:", missing_targets)

if duplicate_groups != 0 or missing_targets != 0:
    raise ValueError("Triage validation failed")

# Save Gold Delta table
df_triage.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_ed_triage_performance")

print("Gold triage performance table saved successfully!")


StatementMeta(, fe278902-f976-4def-994a-ba66bed75f22, 7, Finished, Available, Finished, False)

Duplicate triage groups: 0
Missing treatment targets: 0
Gold triage performance table saved successfully!


In [6]:

df_check = spark.table("gold_ed_triage_performance")

print("Total triage records:", df_check.count())
print("Total columns:", len(df_check.columns))

print(
    "Duplicate triage groups:",
    df_check.groupBy(
        "facility_code",
        "quarter_end_month",
        "triage_category"
    ).count().filter("count > 1").count()
)

print(
    "Missing treatment targets:",
    df_check.filter(
        "recommended_time_minutes IS NULL"
    ).count()
)


StatementMeta(, fe278902-f976-4def-994a-ba66bed75f22, 8, Finished, Available, Finished, False)

Total triage records: 5775
Total columns: 14
Duplicate triage groups: 0
Missing treatment targets: 0


In [1]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Select September 2025 data for six hospitals
peer_codes = [104, 141, 172, 49, 30, 16]

triage_data = (
    spark.table("gold_ed_triage_performance")
    .filter(F.col("quarter_end_month") == F.lit("2025-09-01"))
    .filter(F.col("facility_code").isin(peer_codes))
)

# Calculate common reference triage weights
reference = (
    triage_data
    .groupBy("triage_category")
    .agg(F.sum("attendances").alias("category_total"))
)

total_attendances = reference.agg(
    F.sum("category_total")
).first()[0]

reference = reference.withColumn(
    "triage_weight",
    F.col("category_total") / F.lit(total_attendances)
)

# Apply common weights to each hospital's triage performance
adjusted = (
    triage_data
    .join(
        reference.select("triage_category", "triage_weight"),
        "triage_category"
    )
    .groupBy("facility_code")
    .agg(
        F.sum(
            F.col("seen_within_target_pct") *
            F.col("triage_weight")
        ).alias("triage_adjusted_timeliness_pct"),
        F.count("seen_within_target_pct")
         .alias("valid_categories")
    )
)

# Join published hospital-level metrics
hospital = (
    spark.table("gold_ed_hospital_quarterly")
    .filter(F.col("quarter_end_month") == F.lit("2025-09-01"))
    .filter(F.col("facility_code").isin(peer_codes))
)

df_benchmark = (
    hospital
    .join(adjusted, "facility_code", "inner")
    .withColumn(
        "triage_adjusted_timeliness_pct",
        F.when(
            F.col("valid_categories") == 5,
            F.round("triage_adjusted_timeliness_pct", 2)
        )
    )
    .withColumn(
        "adjustment_difference_pp",
        F.round(
            F.col("triage_adjusted_timeliness_pct") -
            F.col("seen_within_target_pct"),
            2
        )
    )
)

# Ensure all six hospitals have complete data
if df_benchmark.count() != 6:
    raise ValueError("Unexpected number of peer hospitals")

if df_benchmark.filter(
    F.col("valid_categories") != 5
).count() > 0:
    raise ValueError("Incomplete triage performance data")

# Save Gold benchmarking table
df_benchmark.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_ed_hospital_comparison")

print("Gold hospital comparison table saved successfully!")

display(
    df_benchmark.select(
        "facility_name",
        "total_attendances",
        "seen_within_target_pct",
        "triage_adjusted_timeliness_pct",
        "adjustment_difference_pp"
    ).orderBy(F.desc("triage_adjusted_timeliness_pct"))
)


StatementMeta(, 180515c4-952d-45b4-91ca-eaa8b6f4da5e, 3, Finished, Available, Finished, False)

Gold hospital comparison table saved successfully!


SynapseWidget(Synapse.DataFrame, 743ef429-6824-44cd-8ad7-0f496ac7f3e6)